<a href="https://colab.research.google.com/github/anokhina-rgb/Google-Colabs/blob/main/Simplemma_Colab_Notebook_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- КРОК 1: ВСТАНОВЛЕННЯ ---
!pip install simplemma python-docx openpyxl

import simplemma
import pandas as pd
from docx import Document
from google.colab import files
from collections import Counter
import re

# --- КРОК 2: КОНФІГУРАЦІЯ ТА СТОП-СЛОВА ---
LANGUAGES = ('en', 'uk', 'de')

# Базовий список стоп-слів (службові частини мови)
STOP_WORDS = {
    # Українська
    'і', 'й', 'та', 'але', 'чи', 'або', 'що', 'як', 'це', 'на', 'в', 'у', 'до', 'за', 'з', 'із', 'зі', 'під', 'над', 'біля',
    'він', 'вона', 'воно', 'вони', 'ми', 'ви', 'я', 'ти', 'мій', 'твій', 'його', 'її', 'наш', 'ваш', 'їх', 'хто', 'що',
    'не', 'ні', 'так', 'б', 'би', 'ж', 'же', 'то', 'ось', 'тільки', 'лише', 'при', 'про', 'від', 'для',
    # English
    'the', 'a', 'an', 'and', 'or', 'but', 'if', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about',
    'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down',
    'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why',
    'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own',
    'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', 'should', 'now', 'i', 'me', 'my', 'myself',
    'we', 'our', 'ours', 'ourselves', 'you', 'your', 'yours', 'he', 'him', 'his', 'she', 'her', 'it', 'its', 'they', 'them',
    # German
    'der', 'die', 'das', 'und', 'in', 'zu', 'den', 'von', 'mit', 'für', 'auf', 'ist', 'im', 'dem', 'nicht', 'ein', 'eine',
    'als', 'auch', 'es', 'an', 'werden', 'aus', 'er', 'hat', 'dass', 'sie', 'nach', 'wird', 'bei', 'einer', 'um', 'am',
    'sich', 'vor', 'noch', 'wie', 'durch', 'man', 'sind', 'einen', 'ab', 'bis', 'ohne', 'gegen'
}

def clean_and_lemmatize(text, langs=LANGUAGES):
    """
    Токенізація, очищення від службових слів та лематизація.
    """
    # Отримуємо токени
    raw_tokens = simplemma.simple_tokenizer(text)

    clean_lemmas = []
    full_mapping = []

    for token in raw_tokens:
        # Очищення від пунктуації та цифр
        clean_token = re.sub(r'[^\w\s]', '', token)
        if not clean_token or clean_token.isdigit() or len(clean_token) < 2:
            continue

        lemma = simplemma.lemmatize(clean_token, lang=langs)
        low_lemma = lemma.lower()

        # Перевірка на стоп-слова
        if low_lemma not in STOP_WORDS:
            clean_lemmas.append(low_lemma)
            full_mapping.append({
                'Token': token,
                'Lemma': low_lemma
            })

    return clean_lemmas, full_mapping

def get_ngrams(lemmas, n=2):
    """
    Створення n-грам (мультикомпонентних одиниць) з послідовності лем.
    """
    return [" ".join(lemmas[i:i+n]) for i in range(len(lemmas)-n+1)]

# --- КРОК 3: ЗАВАНТАЖЕННЯ ТА ОБРОБКА ---
print("Завантажте файл (.txt) для пошуку мультикомпонентних одиниць:")
uploaded = files.upload()

if not uploaded:
    print("Файл не обрано. Використовуємо вбудований приклад.")
    input_corpus = """
    Strategic partnership between Ukraine and the US is crucial.
    The President of Ukraine discussed military aid.
    New atmospheric conditions were observed by researchers.
    Стратегічне партнерство між Україною та США є критично важливим.
    Президент України обговорив військову допомогу.
    Нові атмосферні умови були помічені дослідниками.
    """
else:
    file_name = list(uploaded.keys())[0]
    input_corpus = uploaded[file_name].decode('utf-8', errors='ignore')

# Обробка
clean_lemmas, mapping = clean_and_lemmatize(input_corpus)

# Генерація n-грам (біграми та триграми)
bigrams = get_ngrams(clean_lemmas, n=2)
trigrams = get_ngrams(clean_lemmas, n=3)

# Частотний аналіз
bigram_freq = Counter(bigrams).most_common(30)
trigram_freq = Counter(trigrams).most_common(30)
lemma_freq = Counter(clean_lemmas).most_common(30)

# Перетворення в DataFrame для експорту
df_bigrams = pd.DataFrame(bigram_freq, columns=['Bigram', 'Frequency'])
df_trigrams = pd.DataFrame(trigram_freq, columns=['Trigram', 'Frequency'])
df_lemmas = pd.DataFrame(lemma_freq, columns=['Lemma', 'Frequency'])

# --- КРОК 4: ЕКСПОРТ ---

# Excel
with pd.ExcelWriter('multi_component_analysis.xlsx') as writer:
    df_lemmas.to_excel(writer, sheet_name='Lemmas', index=False)
    df_bigrams.to_excel(writer, sheet_name='Bigrams (2-word)', index=False)
    df_trigrams.to_excel(writer, sheet_name='Trigrams (3-word)', index=False)

# Word Звіт
doc = Document()
doc.add_heading('Аналіз мультикомпонентних одиниць (без стоп-слів)', 0)

def add_freq_table(title, data, columns):
    doc.add_heading(title, level=1)
    table = doc.add_table(rows=1, cols=2)
    table.style = 'Table Grid'
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = columns[0]
    hdr_cells[1].text = columns[1]
    for item, freq in data[:20]:
        row_cells = table.add_row().cells
        row_cells[0].text = str(item)
        row_cells[1].text = str(freq)

add_freq_table('Топ Біграми (n=2)', bigram_freq, ['Сполука', 'Частота'])
add_freq_table('Топ Триграми (n=3)', trigram_freq, ['Сполука', 'Частота'])

doc.save('multi_component_report.docx')

print("\n--- ТОП БІГРАМИ (БЕЗ СЛУЖБОВИХ СЛІВ) ---")
print(df_bigrams.head(10))

print("\nФайли 'multi_component_analysis.xlsx' та 'multi_component_report.docx' готові.")
# files.download('multi_component_analysis.xlsx')
# files.download('multi_component_report.docx')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.6 MB/s eta 0:00:00
Завантажте файл (.txt) для пошуку мультикомпонентних одиниць:


Saving zelensky_hails_new_ideas_on_peace_after_talk_with_.txt to zelensky_hails_new_ideas_on_peace_after_talk_with_.txt

--- ТОП БІГРАМИ (БЕЗ СЛУЖБОВИХ СЛІВ) ---
            Bigram  Frequency
0          have be          5
1         us envoy          4
2     zelensky say          4
3  ukrainian troop          4
4   donetsk region          4
5         new idea          3
6    zelensky have          3
7    zelensky hail          2
8         hail new          2
9       idea peace          2

Файли 'multi_component_analysis.xlsx' та 'multi_component_report.docx' готові.
